# Swap networks

Adapted from Qualtran v0.7.0, Copyright 2023–2024 Google LLC, Apache-2.0.

In [ ]:
from collections import Counter
from sympy import Symbol, symbols
from IPython.display import SVG, display
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.circuit import Circuit
from flagquantum.compiler._component_trace.resource_model import estimate_component_resources
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

def show_graph(graph):
    display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
    display(graph.leaf_counts)

def emitted(op):
    circuit = Circuit(op.qubit_count)
    circuit.append(op)
    return {"reserved wires": circuit.n_qubits, "emitted gates": Counter(
        gate.name for gate in circuit.to_ir().instructions
    )}

## Relative-phase controlled exchange

The through ports are `ctrl` and two big-endian words `x` and `y`. Control one exchanges the words. The circuit preserves the source's basis-dependent phases. It is self-adjoint and uses $4n$ T-family gates for $n$ bits per word. The call graph retains the separate one-wire and two-wire Clifford declarations.

- [Low, Kliuchnikov and Schaeffer, Appendix B.2.c](https://arxiv.org/abs/1812.00954)

In [ ]:
from examples.algorithms.swap_network_catalog import (
    RelativeSwapExample, relative_swap_call_graph, relative_swap_logical_width, relative_swap_signature,
)

### Example Instances

In [ ]:
n = Symbol('n')
relative_symbolic = relative_swap_signature(n)

In [ ]:
relative_small = RelativeSwapExample(4)

In [ ]:
relative_large = RelativeSwapExample(64)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Symbolic relative swap", relative_symbolic, {}),
    ("Four-bit relative swap", relative_small.signature, {}),
    ("64-bit relative swap", relative_large.signature, {}),
])))

### Call Graph

In [ ]:
show_graph(relative_swap_call_graph(n))
display({'symbolic logical wires': relative_swap_logical_width(n)})
display(emitted(relative_small))

## Swap with zero

The address selects a row to move into the first row of the target array. Multi-axis addresses are big-endian words and targets use row-major order. The source network traverses the innermost axis first. It retains the relative phases of its constituent exchanges. The inverse reverses the exchange order. The source declares one fewer exchange than the number of target rows.

- [Low, Kliuchnikov and Schaeffer, SelectSwap](https://arxiv.org/abs/1812.00954)

In [ ]:
from examples.algorithms.swap_network_catalog import (
    SwapWithZeroExample, swap_with_zero_call_graph, swap_with_zero_signature,
)

### Example Instances

In [ ]:
swz = SwapWithZeroExample((8,), 32, (4,))

In [ ]:
swz_small = SwapWithZeroExample((3,), 2, (2,))

In [ ]:
swz_multi = SwapWithZeroExample((4, 3), 2, (15, 5))

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Four 32-bit rows", swz.signature, {}),
    ("Two two-bit rows", swz_small.signature, {}),
    ("15 by 5 by two-bit rows", swz_multi.signature, {}),
])))

### Call Graph

In [ ]:
show_graph(swap_with_zero_call_graph(swz.word, swz.shape))
display(emitted(swz_small))

## Multiplexed controlled exchange

A bounded unary traversal visits the in-range addresses and swaps the addressed target row with the extra `output` word. The leaf is an exact controlled exchange, with input and output words both through. The traversal reuses clean work and measurement slots. The padded address behavior follows the emitted traversal and is outside the bounded selection contract.

- [Su et al., Fault-tolerant quantum simulations of chemistry in first quantization](https://arxiv.org/abs/2105.12767)

In [ ]:
from examples.algorithms.swap_network_catalog import MultiplexedSwapExample

### Example Instances

In [ ]:
multiplexed = MultiplexedSwapExample(3, 5, 2)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Multiplexed exchange", multiplexed.signature, {}),
])))

### Call Graph

In [ ]:
show_graph(multiplexed.call_graph())
display(estimate_component_resources(multiplexed.resource_model()))
display(emitted(multiplexed))

In [ ]:
swz_other = SwapWithZeroExample((2, 2), 2, (3, 4))
display(emitted(swz_other))

In [ ]:
selection = symbols("p q r")
word = Symbol("b")
shape = symbols("P Q R")
display(swap_with_zero_signature(selection, word, shape))
display(swap_with_zero_call_graph(word, shape).leaf_counts)